# Hex Maze Session Inventory 

In [32]:
import numpy as np
import pandas as pd
import datajoint as dj

import spyglass.common as sgc

# Hex maze behavior tables
from spyglass_hexmaze.hex_maze_behavior import (
    HexMazeBlock,
    HexCentroids,
    HexPositionSelection,
    HexPosition,
)
# Hex maze decode tables
from spyglass_hexmaze.hex_maze_decoding import (
    HexMazeDecodedPosition,
    HexMazeDecodedPositionHex,
    HexMazeDecodedHexPath,
)


## Find all sessions in `HexMazeBlock`

In [38]:
# Get all sessions in HexMazeBlock
hex_maze_sessions = sorted(set(HexMazeBlock.fetch('nwb_file_name')))
session_keys = [{'nwb_file_name': s} for s in hex_maze_sessions]

print(f'Found {len(hex_maze_sessions)} hex maze sessions:')
for s in hex_maze_sessions:
    print('   ', s)
    
# Get lab + subject_id for each hex maze session
session_lab = pd.DataFrame((sgc.Session & session_keys).fetch('nwb_file_name', 'subject_id', 'lab_name',
                                                           as_dict=True))

# Get task type
blocks = pd.DataFrame(HexMazeBlock.fetch('nwb_file_name', 'epoch', 'task_type', as_dict=True))
epoch_task = (
    blocks.groupby(['nwb_file_name', 'epoch'])['task_type']
    .agg(lambda x: ', '.join(sorted(set(x))))  # single task type per epoch
    .reset_index()
)

# Attach lab + subject to every epoch
epoch_task = epoch_task.merge(session_lab, on='nwb_file_name', how='left')

# Flag each epoch as barrier vs probability change
epoch_task['is_barrier'] = epoch_task['task_type'].str.contains('barrier', case=False)
epoch_task['is_prob'] = epoch_task['task_type'].str.contains('prob', case=False)

n_sessions = epoch_task['nwb_file_name'].nunique()
n_epochs = len(epoch_task)
print(f'Total hex maze sessions: {n_sessions}')
print(f'Total hex maze epochs:   {n_epochs}')
print(f'Total unique subjects:   {epoch_task["subject_id"].nunique()}')

# Breakdown by lab (sessions, epochs, subjects, task type)
lab_summary = epoch_task.groupby('lab_name').agg(
    n_subjects=('subject_id', 'nunique'),
    n_sessions=('nwb_file_name', 'nunique'),
    n_epochs=('epoch', 'size'),
    n_barrier=('is_barrier', 'sum'),
    n_prob=('is_prob', 'sum'),
)
print('\n By lab: ')
print(lab_summary.to_string())

# Breakdown by subject
subj_summary = epoch_task.groupby('subject_id').agg(
    n_sessions=('nwb_file_name', 'nunique'),
    n_epochs=('epoch', 'size'),
    n_barrier=('is_barrier', 'sum'),
    n_prob=('is_prob', 'sum'),
)
print('\nBy subject')
print(subj_summary.to_string())

# Breakdown by task type
print('\nEpochs by task type')
print(epoch_task['task_type'].value_counts().to_string())

Found 82 hex maze sessions:
    BraveLu20240516_.nwb
    BraveLu20240518_.nwb
    BraveLu20240519_.nwb
    BraveLu20240615_.nwb
    BraveLu20240617_.nwb
    BraveLu20240619_.nwb
    BraveLu20240622_.nwb
    IM-1478_20220719_.nwb
    IM-1478_20220720_.nwb
    IM-1478_20220724_.nwb
    IM-1478_20220725_.nwb
    IM-1478_20220726_.nwb
    IM-1478_20220727_.nwb
    IM-1594_20230725_.nwb
    IM-1594_20230726_.nwb
    IM-1594_20230727_.nwb
    IM-1594_20230728_.nwb
    IM-1830_pacquiao_20250217_.nwb
    IM-1830_pacquiao_20250224_.nwb
    IM-1830_pacquiao_20250226_.nwb
    IM-1830_pacquiao_20250227_.nwb
    IM-1830_pacquiao_20250228_.nwb
    IM-1830_pacquiao_20250407_.nwb
    IM-1830_pacquiao_20250408_.nwb
    IM-1830_pacquiao_20250411_.nwb
    IM-1830_pacquiao_20250414_.nwb
    IM-1830_pacquiao_20250417_.nwb
    IM-1844_elsa_20250408_.nwb
    IM-1844_elsa_20250414_.nwb
    IM-1844_elsa_20250418_.nwb
    IM-1844_elsa_20250423_.nwb
    IM-1844_elsa_20250424_.nwb
    IM-1844_elsa_20250425_.nwb
 